In [75]:
import os
import torch
import pickle
import re
import numpy as np
from tqdm import tqdm
from transformers import T5Tokenizer, T5EncoderModel

# ================= 配置区域 =================
FASTA_INPUT = "./data_input/5551_add_515_872.fasta"
OUTPUT_PKL = "prot5_5551_add_515_872.pkl"
PROTT5_PATH = "/home/gaozhw/encode_model/prot_t5_xl_uniref50"

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 1  
# ===========================================

def parse_fasta(file_path):
    seq_dict = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        sid, seq = None, []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if sid: seq_dict[sid] = "".join(seq)
                sid = line.split()[0].replace(">", "")
                seq = []
            else: seq.append(line)
        if sid: seq_dict[sid] = "".join(seq)
    return seq_dict

def main():
    # 1. 加载模型
    print(f"🚀 正在加载 ProtT5 模型: {PROTT5_PATH}")
    tokenizer = T5Tokenizer.from_pretrained(PROTT5_PATH, do_lower_case=False)
    model = T5EncoderModel.from_pretrained(PROTT5_PATH).to(DEVICE)
    model.eval()

    # 2. 读取序列
    seq_dict = parse_fasta(FASTA_INPUT)
    ids = list(seq_dict.keys())
    # 预处理：T5 需要序列间有空格，并将特殊氨基酸替换为 X
    sequences = [" ".join(list(re.sub(r"[UZOB]", "X", seq_dict[sid]))) for sid in ids]

    print(f"📊 待编码序列总数: {len(ids)}")
    
    encoded_data = []

    # 3. 分批提取特征
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc="Encoding ProtT5"):
            batch_seqs = sequences[i : i + BATCH_SIZE]
            batch_ids = ids[i : i + BATCH_SIZE]

            # Tokenize
            inputs = tokenizer.batch_encode_plus(
                batch_seqs, 
                add_special_tokens=True, 
                padding=True, 
                return_tensors="pt"
            ).to(DEVICE)

            # 推理
            outputs = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
            last_hidden_states = outputs.last_hidden_state

            # 计算 Mean Pooling (去除 Padding 部分的影响)
            for j, mask in enumerate(inputs['attention_mask']):
                valid_len = mask.sum().item()
                # 去掉结尾的结束符 token (EOS)
                actual_emb = last_hidden_states[j, :valid_len-1, :].mean(dim=0)
                
                # 核心：确保保存为 float32 的 Numpy 数组，彻底规避 dict 报错问题
                encoded_data.append({
                    'id': batch_ids[j],
                    'embedding': actual_emb.cpu().numpy().astype(np.float32)
                })

    # 4. 保存结果
    with open(OUTPUT_PKL, 'wb') as f:
        pickle.dump(encoded_data, f)
    
    print(f"\n✅ 编码完成！结果已保存至: {OUTPUT_PKL}")
    print(f"💡 提取特征维度: {encoded_data[0]['embedding'].shape}")

if __name__ == "__main__":
    main()

🚀 正在加载 ProtT5 模型: /home/gaozhw/encode_model/prot_t5_xl_uniref50
📊 待编码序列总数: 6938


Encoding ProtT5: 100%|██████████| 6938/6938 [18:40<00:00,  6.19it/s]  


✅ 编码完成！结果已保存至: prot5_5551_add_515_872.pkl
💡 提取特征维度: (1024,)


In [59]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# ================= 配置区域 =================
SEEDS = [42, 2023, 1234, 567, 100]
EMBEDDING_PKL = 'prot5_5551_add_515_872.pkl' 
MODEL_PATH_TEMPLATE = "model_seed_{}.pth"
OUTPUT_CSV = "predictions_5551_ensemble.csv"

INPUT_DIM = 1024 
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# ===========================================

class FinalSelectionMLP(nn.Module):
    def __init__(self, input_dim):
        super(FinalSelectionMLP, self).__init__()
        # 匹配你之前报错信息中确认的结构 (256 -> 64 -> 1)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 64),  nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x): return self.net(x)

def main():
    # 1. 加载并自动探测数据格式
    print(f">>> 正在读取特征文件: {EMBEDDING_PKL}")
    if not os.path.exists(EMBEDDING_PKL):
        print(f"❌ 错误: 找不到文件 {EMBEDDING_PKL}")
        return

    with open(EMBEDDING_PKL, 'rb') as f:
        emb_data = pickle.load(f)

    # --- 自动检测逻辑 ---
    if isinstance(emb_data, list):
        print(f"✅ 检测到 List 格式 (长度: {len(emb_data)})")
        # 获取第一个样本的所有键
        sample_keys = list(emb_data[0].keys())
        print(f"🔍 样本包含的键: {sample_keys}")
        
        # 自动定位特征键：排除 id 和 label 后剩下的第一个 1024 维特征
        feat_key = None
        for k in sample_keys:
            if k not in ['id', 'label'] and isinstance(emb_data[0][k], (np.ndarray, list)):
                if len(emb_data[0][k]) == INPUT_DIM:
                    feat_key = k
                    break
        
        if not feat_key:
            raise KeyError("无法在数据中自动识别出 1024 维的特征向量，请检查特征 Key 名称。")
            
        print(f"🎯 自动选定特征键: '{feat_key}'")
        ids = [item['id'] for item in emb_data]
        X_raw = np.array([item[feat_key] for item in emb_data], dtype=np.float32)
    else:
        print("✅ 检测到 Dict 格式")
        ids = list(emb_data.keys())
        X_raw = np.array([emb_data[sid] for sid in ids], dtype=np.float32)

    # 2. 数据标准化 (重要：必须与训练时的量纲保持一致)
    print("⚖️ 正在进行数据标准化...")
    scaler = StandardScaler()
    X_features = scaler.fit_transform(X_raw)

    # 3. 集成预测
    all_preds = np.zeros((len(ids), len(SEEDS)))
    
    for idx, seed in enumerate(SEEDS):
        model_file = MODEL_PATH_TEMPLATE.format(seed)
        if not os.path.exists(model_file):
            print(f"⚠️ 跳过 Seed {seed}: 找不到权重文件")
            continue
            
        print(f"🚀 正在加载模型: {model_file}")
        model = FinalSelectionMLP(INPUT_DIM).to(DEVICE)
        # 使用非严格模式加载，以防结构有细微差异
        model.load_state_dict(torch.load(model_file, map_location=DEVICE))
        model.eval()
        
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_features).to(DEVICE)
            logits = model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            all_preds[:, idx] = probs
            
        del model
        torch.cuda.empty_cache()

    # 4. 汇总结果
    avg_probs = np.mean(all_preds, axis=1)
    results_df = pd.DataFrame({
        'ID': ids,
        'Prob_Avg': avg_probs,
        'Prediction': (avg_probs > 0.5).astype(int)
    })
    
    for i, seed in enumerate(SEEDS):
        results_df[f'Prob_Seed_{seed}'] = all_preds[:, i]

    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✨ 处理完成！结果已保存至: {OUTPUT_CSV}")
    print(f"📊 预测统计: 正样本 = {results_df['Prediction'].sum()}, 负样本 = {len(ids) - results_df['Prediction'].sum()}")

if __name__ == "__main__":
    main()

>>> 正在读取特征文件: prot5_5551_add_515_872.pkl
✅ 检测到 List 格式 (长度: 6938)
🔍 样本包含的键: ['id', 'embedding']
🎯 自动选定特征键: 'embedding'
⚖️ 正在进行数据标准化...
🚀 正在加载模型: model_seed_42.pth
🚀 正在加载模型: model_seed_2023.pth
🚀 正在加载模型: model_seed_1234.pth
🚀 正在加载模型: model_seed_567.pth
🚀 正在加载模型: model_seed_100.pth

✨ 处理完成！结果已保存至: predictions_5551_ensemble.csv
📊 预测统计: 正样本 = 2593, 负样本 = 4345


In [22]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# ================= 配置区域 =================
PRED_CSV = "predictions_5551_ensemble.csv"
FASTA_FILES = ["cat_0.4.fasta", "cat_0.5.fasta", "cat_0.6.fasta", "cat_0.7.fasta"]
K_GAP = 5  # CKSAAP K=5

# 20种标准氨基酸
AA = 'ACDEFGHIKLMNPQRSTVWY'
# 生成 AA 对组合 (400种)
AA_PAIRS = [a + b for a in AA for b in AA]
# ===========================================

def parse_fasta(file_path):
    """解析 FASTA 文件获取 ID 和序列"""
    seq_dict = {}
    if not os.path.exists(file_path):
        print(f"⚠️ 找不到文件: {file_path}")
        return seq_dict
    with open(file_path, 'r') as f:
        seq_id, sequence = None, []
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith('>'):
                if seq_id: seq_dict[seq_id] = "".join(sequence)
                seq_id = line.split()[0].replace('>', '')
                sequence = []
            else: sequence.append(line)
        if seq_id: seq_dict[seq_id] = "".join(sequence)
    return seq_dict

def calculate_cksaap(sequence, k=5):
    """计算单条序列的 CKSAAP 特征"""
    n = len(sequence)
    if n <= k + 1:
        return np.zeros(400)
    
    # 统计 400 种组合出现的频次
    count_dict = {pair: 0 for pair in AA_PAIRS}
    total_pairs = n - k - 1
    
    for i in range(total_pairs):
        aa_i = sequence[i]
        aa_ik = sequence[i + k + 1]
        pair = aa_i + aa_ik
        if pair in count_dict:
            count_dict[pair] += 1
            
    # 转换为频率向量
    feature_vector = np.array([count_dict[pair] / total_pairs for pair in AA_PAIRS])
    return feature_vector

def main():
    # 1. 加载预测标签
    print(f">>> 正在读取预测结果: {PRED_CSV}")
    pred_df = pd.read_csv(PRED_CSV)
    # 将 ID 转为字符串方便匹配，建立 ID -> Label 的映射
    label_map = dict(zip(pred_df['ID'].astype(str), pred_df['Prediction']))

    # 2. 循环处理每个 FASTA 文件
    for fasta_file in FASTA_FILES:
        print(f"\n🚀 正在处理文件: {fasta_file}")
        seq_dict = parse_fasta(fasta_file)
        
        if not seq_dict:
            continue

        data_list = []
        for sid, seq in tqdm(seq_dict.items(), desc="编码进度"):
            # 获取标签 (如果 ID 不在预测结果中，默认标记为 0 或跳过)
            label = label_map.get(str(sid), None)
            if label is None:
                continue # 如果找不到对应的预测标签，跳过该条序列
            
            # 执行 CKSAAP 编码
            ck_feat = calculate_cksaap(seq.upper(), k=K_GAP)
            
            # 构造字典：ID + Label + 400个特征维度
            row = {'ID': sid, 'Label': label}
            for i, val in enumerate(ck_feat):
                row[f'CK_pos_{i}'] = val
            
            data_list.append(row)
        
        # 3. 转换为 DataFrame 并导出
        if data_list:
            output_name = fasta_file.replace('.fasta', '_CKSAAP_K5.csv')
            result_df = pd.DataFrame(data_list)
            result_df.to_csv(output_name, index=False)
            print(f"✅ 已生成带标签的编码文件: {output_name} (样本数: {len(result_df)})")
        else:
            print(f"❌ 文件 {fasta_file} 中没有匹配到预测标签的序列。")

if __name__ == "__main__":
    main()

>>> 正在读取预测结果: predictions_5551_ensemble.csv

🚀 正在处理文件: cat_0.4.fasta


编码进度:   0%|          | 0/958 [00:00<?, ?it/s]

编码进度: 100%|██████████| 958/958 [00:00<00:00, 4415.14it/s]


✅ 已生成带标签的编码文件: cat_0.4_CKSAAP_K5.csv (样本数: 958)

🚀 正在处理文件: cat_0.5.fasta


编码进度: 100%|██████████| 1488/1488 [00:00<00:00, 4531.48it/s]


✅ 已生成带标签的编码文件: cat_0.5_CKSAAP_K5.csv (样本数: 1488)

🚀 正在处理文件: cat_0.6.fasta


编码进度: 100%|██████████| 2018/2018 [00:00<00:00, 4622.07it/s]


✅ 已生成带标签的编码文件: cat_0.6_CKSAAP_K5.csv (样本数: 2018)

🚀 正在处理文件: cat_0.7.fasta


编码进度: 100%|██████████| 2635/2635 [00:00<00:00, 4597.36it/s]


✅ 已生成带标签的编码文件: cat_0.7_CKSAAP_K5.csv (样本数: 2635)
